# Properties of the Derivative — The Chain Rule

A guided notebook transcribed and expanded from the Coursera
*Calculus for Machine Learning and Data Science* lecture on the **chain rule**
(Week 1 — Derivatives and Optimization).

This is the last (and most powerful) of the derivative rules — for **compositions** of
functions.

This notebook explains:

- the rule in Leibniz form: $\dfrac{dg}{dt} = \dfrac{dg}{dh}\cdot\dfrac{dh}{dt}$
- the Lagrange form $(g\circ h)'(t) = g'\big(h(t)\big)\cdot h'(t)$ — **mind the input!**
- chaining more functions, with the mountain/car (temperature–height–time) intuition
- the $\Delta$ derivation (cancel $\Delta h$, take the limit), SymPy and numerical checks

This notebook follows the repository `GUIDELINES.md` template and continues directly from
*Properties of the Derivative — The Product Rule*.

## 1. The rule (Leibniz notation)

Suppose you apply $h$ to $t$, then apply $g$ to the result — a **composition** $g(h(t))$. To
differentiate it, **multiply** the two rates of change:

$$
\frac{dg}{dt} = \frac{dg}{dh}\cdot\frac{dh}{dt}.
$$

In Leibniz notation this looks like fractions where $dh$ "cancels" — a useful (if informal)
memory aid.

### Why "chain"?

You can keep composing. For $f\big(g(h(t))\big)$:

$$
\frac{df}{dt} = \frac{df}{dg}\cdot\frac{dg}{dh}\cdot\frac{dh}{dt}.
$$

Each extra function adds another factor to the product.

## 2. Lagrange notation — mind the input

In Lagrange notation there's a subtlety:

- $\dfrac{dh}{dt} = h'(t)$,
- $\dfrac{dg}{dh}$ is **not** $g'(t)$ — it is $g'$ evaluated at $g$'s input, i.e. $g'\big(h(t)\big)$.

So for two functions:

$$
\boxed{\;\big(g\circ h\big)'(t) = g'\big(h(t)\big)\cdot h'(t).\;}
$$

And for three:

$$
\big(f\circ g\circ h\big)'(t) = f'\big(g(h(t))\big)\cdot g'\big(h(t)\big)\cdot h'(t).
$$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

t = sp.symbols('t')
h = sp.Function('h'); g = sp.Function('g')

# SymPy applies the chain rule automatically
print("d/dt g(h(t)) =", sp.diff(g(h(t)), t))   # Derivative(g(h(t)), h(t)) * h'(t)

d/dt g(h(t)) = Derivative(g(h(t)), h(t))*Derivative(h(t), t)


## 3. The mountain/car intuition

You drive **up a mountain**: hot at the base, cold at the top.

- Temperature changes with **height**: $\dfrac{dT}{dh}$.
- Height changes with **time** as you drive: $\dfrac{dh}{dt}$.
- So temperature changes with **time**: $\dfrac{dT}{dt}$.

The chain rule says the third is the product of the first two:

$$
\frac{dT}{dt} = \frac{dT}{dh}\cdot\frac{dh}{dt}.
$$

In [2]:
# concrete numbers: climb at 1000 m per unit time; temperature drops 6°C per 1000 m
h_of_t = 1000*t                     # height(time)
T_of_h = 30 - sp.Rational(6,1000)*sp.symbols('hh')   # temperature(height), lapse rate
hh = sp.symbols('hh')

dh_dt = sp.diff(h_of_t, t)                 # 1000
dT_dh = sp.diff(T_of_h, hh)                # -6/1000
dT_dt = dT_dh * dh_dt                       # chain rule
print(f"dh/dt = {dh_dt}")
print(f"dT/dh = {dT_dh}")
print(f"dT/dt = (dT/dh)(dh/dt) = {dT_dt}  (°C per unit time)")

# cross-check by composing T(h(t)) directly
T_of_t = T_of_h.subs(hh, h_of_t)
print("direct: d/dt T(h(t)) =", sp.diff(T_of_t, t))

dh/dt = 1000
dT/dh = -3/500
dT/dt = (dT/dh)(dh/dt) = -6  (°C per unit time)
direct: d/dt T(h(t)) = -6


## 4. The $\Delta$ derivation

Picture time, height, and temperature on three axes. A small step $\Delta t$ produces a
small $\Delta h$, which produces a small $\Delta T$:

$$
\frac{\Delta T}{\Delta t} = \frac{\Delta T}{\Delta h}\cdot\frac{\Delta h}{\Delta t}
\qquad (\Delta h \text{ cancels — these are just numbers}).
$$

As $\Delta t \to 0$, all the deltas shrink and the ratios become derivatives:

$$
\frac{dT}{dt} = \frac{dT}{dh}\cdot\frac{dh}{dt}.
$$

## 5. Worked examples with SymPy

In [3]:
x = sp.symbols('x')

examples = [
    sp.sin(x**2),          # g=sin, h=x^2  -> cos(x^2)*2x
    sp.exp(3*x),           # -> 3 e^{3x}
    (x**2 + 1)**5,         # -> 5(x^2+1)^4 * 2x
    sp.log(sp.cos(x)),     # -> -tan(x)
]
for expr in examples:
    print(f"d/dx [{expr}] = {sp.simplify(sp.diff(expr, x))}")

d/dx [sin(x**2)] = 2*x*cos(x**2)
d/dx [exp(3*x)] = 3*exp(3*x)
d/dx [(x**2 + 1)**5] = 10*x*(x**2 + 1)**4
d/dx [log(cos(x))] = -tan(x)


### Step-by-step for $\sin(x^2)$

Outer $g(u) = \sin u$ with $g'(u) = \cos u$; inner $h(x) = x^2$ with $h'(x) = 2x$. So

$$
\frac{d}{dx}\sin(x^2) = \cos(x^2)\cdot 2x .
$$

Note we plug the **inner function** $x^2$ into $\cos$, not just $x$.

In [4]:
# numerical check of the chain rule for sin(x^2)
def f(x): return np.sin(x**2)
def chain(x): return np.cos(x**2) * 2*x
def numder(fn, x, eps=1e-6): return (fn(x+eps)-fn(x-eps))/(2*eps)

for x0 in [0.5, 1.0, 1.5]:
    print(f"x={x0}:  numeric={numder(f, x0):.6f},  cos(x²)·2x={chain(x0):.6f}")

x=0.5:  numeric=0.968912,  cos(x²)·2x=0.968912
x=1.0:  numeric=1.080605,  cos(x²)·2x=1.080605
x=1.5:  numeric=-1.884521,  cos(x²)·2x=-1.884521


## 6. Exercises

### Basic
1. State the chain rule for $g(h(x))$ in both Leibniz and Lagrange notation.
2. Why is $\dfrac{d}{dx}g(h(x))$ equal to $g'(h(x))h'(x)$ and not $g'(x)h'(x)$?

### Intermediate
3. Differentiate $f(x) = \cos(3x)$ and $f(x) = e^{x^2}$.
4. Differentiate $f(x) = (2x + 1)^4$.

### Advanced
5. Differentiate the triple composition $f(x) = \sin\!\big(e^{x^2}\big)$.
6. In the mountain analogy, if you climb at $500$ m/min and the temperature drops
   $6.5^\circ$C per $1000$ m, how fast is the temperature changing per minute?

## 7. Solutions / Checks

In [5]:
x = sp.symbols('x')

# 3, 4, 5
print("3. d/dx cos(3x) =", sp.diff(sp.cos(3*x), x), ";  d/dx e^(x²) =", sp.diff(sp.exp(x**2), x))
print("4. d/dx (2x+1)^4 =", sp.diff((2*x+1)**4, x))
print("5. d/dx sin(e^(x²)) =", sp.diff(sp.sin(sp.exp(x**2)), x))

# 6.
dh_dt = 500          # m per min
dT_dh = -6.5/1000    # °C per m
print(f"6. dT/dt = {dT_dh*dh_dt} °C per minute")

print("\n1. Leibniz: dg/dx = (dg/dh)(dh/dt);  Lagrange: g'(h(x))·h'(x).")
print("2. g changes with respect to its own input h(x), so g' must be evaluated at h(x).")

3. d/dx cos(3x) = -3*sin(3*x) ;  d/dx e^(x²) = 2*x*exp(x**2)
4. d/dx (2x+1)^4 = 8*(2*x + 1)**3
5. d/dx sin(e^(x²)) = 2*x*exp(x**2)*cos(exp(x**2))
6. dT/dt = -3.25 °C per minute

1. Leibniz: dg/dx = (dg/dh)(dh/dt);  Lagrange: g'(h(x))·h'(x).
2. g changes with respect to its own input h(x), so g' must be evaluated at h(x).


## 8. Conclusion

- The **chain rule** differentiates compositions by **multiplying** rates of change:
  $\dfrac{dg}{dt} = \dfrac{dg}{dh}\dfrac{dh}{dt}$, i.e. $(g\circ h)'(t) = g'(h(t))\,h'(t)$.
- In Lagrange notation, evaluate the outer derivative at the **inner function's** value
  ($g'(h(t))$, not $g'(t)$).
- It chains to any depth: $f'(g(h))\,g'(h)\,h'$, etc.
- Intuition: temperature changes with time *through* height —
  $\dfrac{dT}{dt} = \dfrac{dT}{dh}\dfrac{dh}{dt}$.
- Together with the scalar, sum, and product rules, the chain rule lets you differentiate
  essentially any function — and it is the engine of **backpropagation** in neural
  networks.

## Appendix — Source transcript

Transcribed with OpenAI Whisper (`small.en`) from a locally downloaded video
(`~/Downloads/index (20).mp4`), the Coursera *Calculus for Machine Learning and Data
Science* (Week 1) lecture **properties of the derivative — the chain rule**. Key spoken
points, lightly cleaned:

> The chain rule is for compositions: apply $h$ to $t$, then $g$ to the output, giving
> $g(h(t))$; its derivative is the product $\frac{dg}{dt} = \frac{dg}{dh}\frac{dh}{dt}$.
> It's called the chain rule because you can keep composing — $f(g(h(t)))$ gives
> $\frac{df}{dg}\frac{dg}{dh}\frac{dh}{dt}$. In Lagrange notation, $\frac{dh}{dt} = h'(t)$,
> but $\frac{dg}{dh} = g'(h(t))$, not $g'(t)$ — you evaluate $g'$ at $g$'s input. Picture
> driving up a mountain: temperature changes with height ($dT/dh$), height changes with
> time ($dh/dt$), so temperature changes with time ($dT/dt = (dT/dh)(dh/dt)$). With small
> steps, $\Delta T/\Delta t = (\Delta T/\Delta h)(\Delta h/\Delta t)$; $\Delta h$ cancels,
> and as $\Delta t \to 0$ these become derivatives — the chain rule.